In [0]:
-- 步骤1: 创建临时视图读取Bronze数据
-- 获取 ADF 传递的参数
-- DECLARE tdspath STRING DEFAULT getArgument('p_tdspath', 'gmall/order_info/default');
CREATE OR REPLACE TEMPORARY VIEW bronze_order_info_raw AS
SELECT 
    *,
    _metadata.file_path as source_file,
    current_timestamp() as load_timestamp
FROM read_files('abfss://bronze@sahyivy.dfs.core.windows.net/'||:p_tdspath||'/*.parquet');

-- 步骤2: 合并新数据到Silver表（UPSERT）
MERGE INTO silver_orders_info AS target
USING bronze_order_info_raw AS source
ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET                
                target.consignee              =source.consignee             ,
                target.consignee_tel          =source.consignee_tel         ,
                target.total_amount           =source.total_amount          ,
                target.order_status           =source.order_status          ,
                target.user_id                =source.user_id               ,
                target.payment_way            =source.payment_way           ,
                target.delivery_address       =source.delivery_address      ,
                target.order_comment          =source.order_comment         ,
                target.out_trade_no           =source.out_trade_no          ,
                target.trade_body             =source.trade_body            ,
                target.create_time            =source.create_time           ,
                target.operate_time           =source.operate_time          ,
                target.expire_time            =source.expire_time           ,
                target.process_status         =source.process_status        ,
                target.tracking_no            =source.tracking_no           ,
                target.parent_order_id        =source.parent_order_id       ,
                target.img_url                =source.img_url               ,
                target.province_id            =source.province_id           ,
                target.activity_reduce_amount =source.activity_reduce_amount,
                target.coupon_reduce_amount   =source.coupon_reduce_amount  ,
                target.original_total_amount  =source.original_total_amount ,
                target.feight_fee             =source.feight_fee            ,
                target.feight_fee_reduce      =source.feight_fee_reduce     ,
                target.refundable_time        =source.refundable_time       ,
                target.source_file            =source.source_file           ,
                target.load_timestamp         =source.load_timestamp        ,
                target.update_timestamp = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (id                    ,
                consignee             ,
                consignee_tel         ,
                total_amount          ,
                order_status          ,
                user_id               ,
                payment_way           ,
                delivery_address      ,
                order_comment         ,
                out_trade_no          ,
                trade_body            ,
                create_time           ,
                operate_time          ,
                expire_time           ,
                process_status        ,
                tracking_no           ,
                parent_order_id       ,
                img_url               ,
                province_id           ,
                activity_reduce_amount,
                coupon_reduce_amount  ,
                original_total_amount ,
                feight_fee            ,
                feight_fee_reduce     ,
                refundable_time       ,
                source_file           ,
                load_timestamp        ,
                update_timestamp
                )
        VALUES (source.id                    ,
                source.consignee             ,
                source.consignee_tel         ,
                source.total_amount          ,
                source.order_status          ,
                source.user_id               ,
                source.payment_way           ,
                source.delivery_address      ,
                source.order_comment         ,
                source.out_trade_no          ,
                source.trade_body            ,
                source.create_time           ,
                source.operate_time          ,
                source.expire_time           ,
                source.process_status        ,
                source.tracking_no           ,
                source.parent_order_id       ,
                source.img_url               ,
                source.province_id           ,
                source.activity_reduce_amount,
                source.coupon_reduce_amount  ,
                source.original_total_amount ,
                source.feight_fee            ,
                source.feight_fee_reduce     ,
                source.refundable_time       ,
                source.source_file           , 
                source.load_timestamp        ,
                current_timestamp()
                );